In [15]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
train_data = pd.read_csv("/content/drive/MyDrive/Week-4/train.csv")
test_data = pd.read_csv("/content/drive/MyDrive/Week-4/test.csv")

<ipython-input-16-139c23d2a60d>:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv("/content/drive/MyDrive/Week-4/train.csv")


In [17]:
test_data['StateHoliday'].unique()

array(['0', 'a'], dtype=object)

In [18]:
# changing date column to datetime data type
train_data['Date'] = pd.to_datetime(train_data['Date'])
test_data['Date'] = pd.to_datetime(test_data['Date'])

# Droping data with missing dates
def drop_missing_dates(df):
    cleaned_data = df.dropna(subset = ['Date'])
    return cleaned_data

train_data = drop_missing_dates(train_data)
test_data = drop_missing_dates(test_data)

In [19]:
# Encodin catagorical column ("State Hoiday")
from sklearn.preprocessing import OneHotEncoder


def encoder_fun(df, column):
    df[column] = df[column].astype(str)
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded= encoder.fit_transform(df[[column]])
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out([column]))
    df_encoded = pd.concat([df.drop(column, axis= 1), encoded_df], axis=1)
    return df_encoded

train_data = encoder_fun(train_data, 'StateHoliday')
test_data = encoder_fun(test_data, 'StateHoliday')


In [20]:
# Filling missing values using simple imputer
imputer = SimpleImputer(strategy='median')
train_imputed = imputer.fit_transform(train_data.drop(['Date'], axis = 1))
test_imputed = imputer.fit_transform(test_data.drop(['Date'], axis = 1))

# changing parrays formed during imputation to dataframe
train_imputed = pd.DataFrame(train_imputed,columns=train_data.drop(['Date'],axis=1).columns)
test_imputed = pd.DataFrame(test_imputed,columns=test_data.drop(['Date'], axis=1).columns)

In [23]:
# merging date column with the imputed ones
train_processed_data = pd.concat([train_data[['Date']], train_imputed], axis=1)
test_processed_data = pd.concat([test_data[['Date']], test_imputed], axis=1)


In [24]:
train_processed_data = train_processed_data.drop(["StateHoliday_b","StateHoliday_c"], axis= 1)

In [27]:
train_processed_data.to_csv("train_processed.csv", index=False)
test_processed_data.to_csv("test_processed.csv", index=False)